# Homework 11: CLIP Image Text Similarity and Zero Shot Classification

This notebook evaluates CLIP in two settings:

1. Caption to image similarity across a balanced sample of 100 CIFAR 10 images from each class.
2. Zero shot classification on CIFAR 10 with per class accuracy and a confusion matrix.

Seed: 2046

In [ ]:
!pip -q install transformers

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision import transforms

SEED = 2046

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Seed:", SEED)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Seed: 2046
Device: cuda
GPU: Tesla T4


## 1. Caption to Image Similarity

CLIP is used to compare a fixed natural language caption against 1,000 CIFAR 10 test images, with exactly 100 images from each class.

Both image and text embeddings are L2 normalized before cosine similarity is computed. The resulting scores are analyzed by CIFAR 10 class and the highest scoring images are visualized.

In [ ]:
CAPTION = "a photo of an airplane flying in the sky"

print("Caption:", CAPTION)

Caption: a photo of an airplane flying in the sky


In [ ]:
text_inputs = processor(
    text=[CAPTION],
    return_tensors="pt",
    padding=True
)

text_inputs = {
    key: value.to(device)
    for key, value in text_inputs.items()
}

with torch.no_grad():
    text_embedding = clip_model.get_text_features(
        **text_inputs
    )

text_embedding = (
    text_embedding
    / text_embedding.norm(
        dim=-1,
        keepdim=True
    )
)

print("Text embedding shape:", tuple(text_embedding.shape))
print(
    "Text embedding norm:",
    text_embedding.norm(dim=-1).item()
)

NameError: name 'processor' is not defined

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(
    CLIP_MODEL
)

clip_model = CLIPModel.from_pretrained(
    CLIP_MODEL
).to(device)

clip_model.eval()

for parameter in clip_model.parameters():
    parameter.requires_grad = False

print("CLIP model:", CLIP_MODEL)
print("Device:", device)
print("CLIP ready.")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model: openai/clip-vit-base-patch32
Device: cuda
CLIP ready.


In [ ]:
required = [
    "processor",
    "clip_model",
    "test_dataset",
    "similarity_indices",
    "CLASS_NAMES",
    "device",
    "CAPTION"
]

for name in required:
    print(name, ":", name in globals())

processor : True
clip_model : True
test_dataset : False
similarity_indices : False
CLASS_NAMES : False
device : True
CAPTION : True


In [ ]:
import numpy as np
from torchvision.datasets import CIFAR10

SEED = 2046
SAMPLES_PER_CLASS = 100

train_dataset = CIFAR10(
    root="/content/data",
    train=True,
    download=True
)

test_dataset = CIFAR10(
    root="/content/data",
    train=False,
    download=True
)

CLASS_NAMES = train_dataset.classes

targets = np.array(test_dataset.targets)

rng = np.random.default_rng(SEED)

similarity_indices = []
class_sample_counts = {}

for class_id, class_name in enumerate(CLASS_NAMES):
    class_indices = np.where(
        targets == class_id
    )[0]

    selected = rng.choice(
        class_indices,
        size=SAMPLES_PER_CLASS,
        replace=False
    )

    similarity_indices.extend(
        selected.tolist()
    )

    class_sample_counts[class_name] = len(selected)

similarity_indices = np.array(
    similarity_indices,
    dtype=int
)

assert len(similarity_indices) == 1000
assert all(
    count == 100
    for count in class_sample_counts.values()
)

print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", CLASS_NAMES)
print("Similarity images:", len(similarity_indices))
print("Per class:", class_sample_counts)
print("CIFAR 10 state restored.")

100%|██████████| 170M/170M [27:01<00:00, 105kB/s]


Train images: 50000
Test images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Similarity images: 1000
Per class: {'airplane': 100, 'automobile': 100, 'bird': 100, 'cat': 100, 'deer': 100, 'dog': 100, 'frog': 100, 'horse': 100, 'ship': 100, 'truck': 100}
CIFAR 10 state restored.


In [ ]:
required = [
    "processor",
    "clip_model",
    "test_dataset",
    "similarity_indices",
    "CLASS_NAMES",
    "device",
    "CAPTION"
]

for name in required:
    print(name, ":", name in globals())

processor : True
clip_model : True
test_dataset : True
similarity_indices : True
CLASS_NAMES : True
device : True
CAPTION : True


In [ ]:
text_inputs = processor(
    text=[CAPTION],
    return_tensors="pt",
    padding=True
)

input_ids = text_inputs["input_ids"].to(device)
attention_mask = text_inputs["attention_mask"].to(device)

with torch.no_grad():
    text_outputs = clip_model.text_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    pooled_text = text_outputs.pooler_output

    text_embedding = clip_model.text_projection(
        pooled_text
    )

text_embedding = (
    text_embedding
    / text_embedding.norm(
        dim=-1,
        keepdim=True
    )
)

print(
    "Text embedding shape:",
    tuple(text_embedding.shape)
)

print(
    "Text embedding norm:",
    text_embedding.norm(
        dim=-1
    ).item()
)

Text embedding shape: (1, 512)
Text embedding norm: 1.0


In [ ]:
import time
import pandas as pd

SIMILARITY_BATCH_SIZE = 64

similarity_records = []

start_time = time.perf_counter()

for start in range(
    0,
    len(similarity_indices),
    SIMILARITY_BATCH_SIZE
):
    batch_indices = similarity_indices[
        start:start + SIMILARITY_BATCH_SIZE
    ]

    batch_images = []
    batch_labels = []

    for index in batch_indices:
        image, label = test_dataset[int(index)]

        batch_images.append(image)
        batch_labels.append(label)

    image_inputs = processor(
        images=batch_images,
        return_tensors="pt"
    )

    pixel_values = image_inputs[
        "pixel_values"
    ].to(device)

    with torch.no_grad():
        vision_outputs = clip_model.vision_model(
            pixel_values=pixel_values
        )

        pooled_images = vision_outputs.pooler_output

        image_embeddings = clip_model.visual_projection(
            pooled_images
        )

    image_embeddings = (
        image_embeddings
        / image_embeddings.norm(
            dim=-1,
            keepdim=True
        )
    )

    similarities = (
        image_embeddings
        @ text_embedding.T
    ).squeeze(1)

    for dataset_index, label, score in zip(
        batch_indices,
        batch_labels,
        similarities.cpu().tolist()
    ):
        similarity_records.append({
            "dataset_index": int(dataset_index),
            "class_id": int(label),
            "class_name": CLASS_NAMES[int(label)],
            "caption": CAPTION,
            "similarity": float(score)
        })

similarity_seconds = (
    time.perf_counter()
    - start_time
)

similarity_df = pd.DataFrame(
    similarity_records
)

print(
    "Images processed:",
    len(similarity_df)
)

print(
    "Similarity runtime seconds:",
    round(similarity_seconds, 2)
)

display(similarity_df.head())

Images processed: 1000
Similarity runtime seconds: 4.17


,dataset_index,class_id,class_name,caption,similarity
0,4474,0,airplane,a photo of an airplane flying in the sky,0.228270
1,7240,0,airplane,a photo of an airplane flying in the sky,0.255243
2,4180,0,airplane,a photo of an airplane flying in the sky,0.270456
3,3338,0,airplane,a photo of an airplane flying in the sky,0.228823
4,4171,0,airplane,a photo of an airplane flying in the sky,0.268674
